In [50]:
text = "Hello, world! This is a simple language model example."

# Get all unique characters
chars = sorted(list(set(text)))
vocab_size = len(chars)

# Create mappings from characters to integers and vice versa
char_to_ix = { ch:i for i,ch in enumerate(chars) }
ix_to_char = { i:ch for i,ch in enumerate(chars) }

# Encode the whole text as a list of integers
data = [char_to_ix[c] for c in text]

print("Encoded:", data)

Encoded: [4, 8, 12, 12, 15, 2, 0, 20, 15, 17, 12, 7, 1, 0, 5, 10, 11, 18, 0, 11, 18, 0, 6, 0, 18, 11, 13, 16, 12, 8, 0, 12, 6, 14, 9, 19, 6, 9, 8, 0, 13, 15, 7, 8, 12, 0, 8, 21, 6, 13, 16, 12, 8, 3]


In [52]:
# Input size (context length)
context_size = 4

X = []  # inputs
Y = []  # targets

for i in range(len(data) - context_size):
    context = data[i:i+context_size]
    target = data[i+context_size]
    X.append(context)
    Y.append(target)

In [53]:
import torch
import torch.nn as nn
import torch.nn.functional as F

X_tensor = torch.tensor(X)
Y_tensor = torch.tensor(Y)

class SimpleLM(nn.Module):
    def __init__(self, vocab_size, context_size, embedding_dim, hidden_dim):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embedding_dim)
        self.fc1 = nn.Linear(context_size * embedding_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x):
        x = self.embed(x)                    # [batch, context, embed]
        x = x.view(x.size(0), -1)            # Flatten
        x = F.relu(self.fc1(x))              # Hidden layer
        logits = self.fc2(x)                 # Output logits
        return logits

In [54]:
model = SimpleLM(vocab_size, context_size, embedding_dim=10, hidden_dim=64)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(1000):
    logits = model(X_tensor)
    loss = loss_fn(logits, Y_tensor)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 3.1430
Epoch 100, Loss: 0.0280
Epoch 200, Loss: 0.0279
Epoch 300, Loss: 0.0278
Epoch 400, Loss: 0.0278
Epoch 500, Loss: 0.0278
Epoch 600, Loss: 0.0278
Epoch 700, Loss: 0.0278
Epoch 800, Loss: 0.0278
Epoch 900, Loss: 0.0278


In [64]:
def generate(model, start_context, length):
    context = [char_to_ix[c] for c in start_context]
    for _ in range(length):
        x = torch.tensor([context[-context_size:]])  
        with torch.no_grad():
            logits = model(x)
            probs = F.softmax(logits, dim=-1)
            next_char_ix = torch.multinomial(probs, num_samples=1).item()
        context.append(next_char_ix)
    return ''.join(ix_to_char[i] for i in context)

print(generate(model, start_context="Hello, world!", length=40))


Hello, world! This is a simple language model example
